# AI Travel Assistant Chatbot - LLaMA 3.1 Fine-Tuning Notebook
## Fine-tune `meta-llama/Llama-3.1-8B-Instruct` on `JasleenSingh91/travel-QA` using QLoRA

---

### What this notebook does (step by step):
| Step | What happens |
|------|--------------|
| 1 | Install required libraries |
| 2 | Connect to Hugging Face with your token |
| 3 | Load and explore the travel Q&A dataset |
| 4 | Convert dataset into chat format |
| 5 | Load the LLaMA base model (4-bit, memory-efficient) |
| 6 | Test base model BEFORE training (for comparison later) |
| 7 | Add LoRA adapters to the model |
| 8 | Train the model on travel data |
| 9 | Plot the training loss graph |
| 10 | Save the LoRA adapter to disk |
| 11 | Test fine-tuned model AFTER training |
| 12 | Side-by-side comparison: Base vs Fine-tuned |
| 13 | (Optional) Push adapter to Hugging Face Hub |

### IMPORTANT - Do this FIRST:
1. Click **Runtime -> Change runtime type**
2. Select **GPU** (T4 is free and works perfectly)
3. Click **Save**
4. Then run cells from top to bottom

---

# THEORY - Understand What You Are Doing
*Read this section before running any code. It explains everything in simple English.*

---

## What is a Base Model?
A **base model** is a very large AI that was already trained on billions of pages of text from the internet.  
Think of it like a very smart student who has read every book in the world but has never studied travel specifically.  
Our base model: `meta-llama/Llama-3.1-8B-Instruct` (8 billion parameters, made by Meta/Facebook).

---

## What is a Dataset?
A **dataset** is a collection of examples used to teach the model.  
Our dataset (`JasleenSingh91/travel-QA`) contains thousands of travel questions and their answers.  
Example:  
- Question: What are the best places to visit in Paris?  
- Answer: Paris is famous for the Eiffel Tower, Louvre Museum, Notre-Dame Cathedral...

---

## What is Fine-Tuning?
**Fine-tuning** means taking the smart base model and training it a little more on our specific travel data.  
It is like sending a smart student to a travel school. They already know everything, now they learn travel expertise on top.  
After fine-tuning, the model gives much better travel answers.

---

## What is 1 Epoch?
An **epoch** means the model has seen every example in the dataset exactly once.  
We start with 1 epoch because it is fast and we can see if training is working.  
More epochs = more learning, but also more time and risk of memorizing instead of understanding.

---

## What is LoRA?
**LoRA** (Low-Rank Adaptation) is a clever trick.  
Instead of changing all 8 billion parameters of the model (expensive!), LoRA adds a tiny set of new layers that learn the travel knowledge.  
The original model stays frozen. Only the tiny LoRA layers are trained.  
This is like adding a travel plugin to a browser - the browser does not change, the plugin adds functionality.

---

## What is QLoRA?
**QLoRA** = LoRA + **Q**uantization.  
Quantization compresses the model from 32-bit numbers to 4-bit numbers, using ~75% less memory.  
This lets you fine-tune LLaMA 8B on a free Colab GPU (16GB) instead of needing expensive hardware.  
QLoRA = affordable fine-tuning without losing much quality.

---

## Why Not Full Fine-Tuning?
Full fine-tuning means updating ALL 8 billion parameters. Problems:  
- **Memory:** Needs 80GB+ GPU RAM (very expensive)  
- **Time:** Takes days  
- **Risk:** Can ruin the model's general knowledge  
LoRA/QLoRA solves all three problems. It is the standard approach used by professionals today.

---

## What is a LoRA Adapter?
The LoRA adapter is the tiny file that contains everything the model learned during fine-tuning.  
It is usually only 10-100 MB (vs 16 GB for the full model).  
You save this adapter, and at inference time you load the base model + adapter together.  
This is what your backend API will load later to answer travel questions.

---

## How to Know Fine-Tuning Worked?
1. **Training loss goes down** - loss starts high (~2.0) and drops (~0.5). Lower = better learning.  
2. **Better answers** - the fine-tuned model gives specific, structured travel answers.  
3. **Comparison test** - ask the same question before and after training and compare.

---

## How to Compare Base Model vs Fine-Tuned Model?
1. Ask the same question to the base model BEFORE training. Save the answer.  
2. Train the model on travel data.  
3. Ask the same question AFTER training. Save the answer.  
4. Print both answers side by side and see the difference.

---

## How to Connect to Backend + Next.js Later?
After saving your LoRA adapter:  
1. **Backend** (FastAPI/Express): Load base model + LoRA adapter, expose a `/chat` API endpoint  
2. **Frontend** (your Next.js app): Send user questions to the backend API, display the answer  
Flow: User types question -> Next.js -> API call -> Python backend -> LLaMA model -> answer -> back to UI

---

# GOOGLE COLAB SETUP CHECKLIST

Before running code, complete these steps:

**Step 1 - Select GPU Runtime:**
- Click **Runtime** (top menu) -> **Change runtime type**
- Under Hardware accelerator select **T4 GPU**
- Click **Save**

**Step 2 - Add Hugging Face Token:**
- Click the key icon on the left sidebar (Secrets)
- Click **+ Add new secret**
- Name: `HF_TOKEN`
- Value: paste your HF token
- Toggle **Notebook access** ON

**Step 3 - Run cells in order:**
- Press the play button on each cell, OR
- Click **Runtime -> Run all** to run everything

> Time estimate: Installation ~5 min | Model loading ~3 min | Training ~30-60 min for 1000 rows on T4

---

In [ ]:
# ==============================================================
# STEP 1: Check GPU and Install Required Libraries
# ==============================================================

import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('GPU is available!')
    print(result.stdout[:500])
else:
    print('No GPU found! Go to Runtime -> Change runtime type -> GPU')
    raise SystemExit('Please enable GPU before continuing.')

In [ ]:
# Install all required libraries
# Unsloth   = optimized training (faster + less memory)
# TRL       = training library for language models
# PEFT      = adds LoRA adapter support
# datasets  = loads data from Hugging Face Hub
# bitsandbytes = enables 4-bit quantization for QLoRA

print('Installing libraries... This takes about 3-5 minutes.')

import subprocess
subprocess.run(['pip', 'install', '-q', 'unsloth'], check=True)
subprocess.run(['pip', 'install', '-q', 'trl', 'transformers', 'datasets',
                'accelerate', 'peft', 'bitsandbytes', 'huggingface_hub'], check=True)
subprocess.run(['pip', 'install', '-q', 'matplotlib'], check=True)

print('All libraries installed successfully!')

In [ ]:
# ==============================================================
# STEP 2: Setup Hugging Face Authentication
# ==============================================================
# IMPORTANT: LLaMA 3.1 is a GATED model.
# Before running this cell you MUST:
#   1. Go to: huggingface.co/meta-llama/Llama-3.1-8B-Instruct
#   2. Log in with your HF account
#   3. Click 'Agree and access repository'
#   4. Wait 2-5 minutes for access to activate
#   5. Then run this cell
import os
from google.colab import userdata
from huggingface_hub import login, whoami

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    HF_TOKEN = os.environ.get("HF_API_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN missing. Add it in Colab Secrets.")

HF_TOKEN = HF_TOKEN.strip().replace("Bearer ", "").replace("\n", "").replace("\r", "")

login(token=HF_TOKEN, add_to_git_credential=False)

print("Logged in as:", whoami()["name"])

# --- Login to Hugging Face ---
from huggingface_hub import login, whoami, model_info

try:
    login(token=HF_TOKEN, add_to_git_credential=False)
except Exception as e:
    raise RuntimeError(
        f'Login failed: {e}\n'
        'Check: Is your token valid? Go to huggingface.co/settings/tokens'
    )

# --- Verify who is logged in ---
try:
    user_info = whoami(token=HF_TOKEN)
    print(f'Logged in as: {user_info["name"]}')
except Exception as e:
    raise RuntimeError(
        f'Token verification failed: {e}\n'
        'Your token may be expired. Get a new one at huggingface.co/settings/tokens'
    )

# --- Check LLaMA model access ---
print('Checking LLaMA 3.1 model access...')
try:
    info = model_info('meta-llama/Llama-3.1-8B-Instruct', token=HF_TOKEN)
    print('LLaMA 3.1-8B-Instruct: ACCESS GRANTED - ready to train!')
except Exception as e:
    err = str(e).lower()
    if any(x in err for x in ['401', 'gated', 'forbidden', 'restricted', '403']):
        raise PermissionError(
            '\n'
            '================================================\n'
            'ACCESS DENIED to LLaMA 3.1-8B-Instruct!\n'
            '================================================\n'
            'YOU MUST DO THIS FIRST:\n'
            '  1. Open this URL in your browser:\n'
            '     https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct\n'
            '  2. Log in with your Hugging Face account\n'
            '  3. Click "Agree and access repository"\n'
            '  4. Wait 2-5 minutes for access to activate\n'
            '  5. Come back to Colab and re-run this cell\n'
            '================================================\n'
        )
    else:
        print(f'Warning: Could not verify model access ({e}). Continuing anyway...')

print('\nAll checks passed! You can now run the next cell.')


In [ ]:
# ==============================================================
# STEP 3: Load and Explore the Travel Dataset
# ==============================================================

from datasets import load_dataset

print('Loading dataset: JasleenSingh91/travel-QA')
print('This might take a minute...\n')

raw_dataset = load_dataset('JasleenSingh91/travel-QA', token=HF_TOKEN)

print('Dataset loaded!')
print('=' * 50)
print('DATASET STRUCTURE:')
print(raw_dataset)

print('\nDATASET SPLITS (train/test/validation):')
for split_name, split_data in raw_dataset.items():
    print(f'  {split_name}: {len(split_data)} rows')

# Use train split if available, otherwise use the first available split
split_key = 'train' if 'train' in raw_dataset else list(raw_dataset.keys())[0]
dataset = raw_dataset[split_key]

print(f'\nCOLUMN NAMES in {split_key!r} split:')
print(dataset.column_names)

print('\nFIRST 3 SAMPLE ROWS:')
print('=' * 50)
for i in range(min(3, len(dataset))):
    print(f'\n--- Row {i + 1} ---')
    for col in dataset.column_names:
        value = str(dataset[i][col])[:300]
        print(f'{col}: {value}')

print('\n' + '=' * 50)
print(f'Dataset has {len(dataset)} total examples')

In [ ]:
# ==============================================================
# STEP 4: Convert Dataset to Chat Format
# ==============================================================
# LLaMA 3.1 expects input in its chat template format.
# We convert each Q&A pair into that format automatically.

# Auto-detect which columns contain the question and answer
QUESTION_CANDIDATES = ['question', 'Question', 'input', 'prompt', 'query', 'text']
ANSWER_CANDIDATES   = ['answer', 'Answer', 'output', 'response', 'completion']

cols = dataset.column_names
question_col = next((c for c in QUESTION_CANDIDATES if c in cols), None)
answer_col   = next((c for c in ANSWER_CANDIDATES   if c in cols), None)

if not question_col or not answer_col:
    print('Could not auto-detect columns. Available columns:', cols)
    print('Manually set the two lines below:')
    question_col = cols[0]  # Change to your question column name
    answer_col   = cols[1]  # Change to your answer column name

print(f'Question column: {question_col!r}')
print(f'Answer column:   {answer_col!r}')


def format_as_chat(example):
    question = str(example[question_col]).strip()
    answer   = str(example[answer_col]).strip()
    text = (
        '<|begin_of_text|>'
        '<|start_header_id|>system<|end_header_id|>\n\n'
        'You are a helpful AI travel assistant. '
        'Provide detailed, accurate, and friendly travel advice.<|eot_id|>'
        '<|start_header_id|>user<|end_header_id|>\n\n'
        f'{question}<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
        f'{answer}<|eot_id|>'
    )
    return {'text': text}


# --- Training size option ---
# USE_SMALL_DATASET = True  -> quick test with 1000 rows (~30-60 min on T4)
# USE_SMALL_DATASET = False -> full dataset training
USE_SMALL_DATASET = True
MAX_ROWS          = 1000

if USE_SMALL_DATASET:
    training_data = dataset.select(range(min(MAX_ROWS, len(dataset))))
    print(f'\nQUICK TEST MODE: Using {len(training_data)} rows')
    print('(Set USE_SMALL_DATASET = False to train on full dataset)')
else:
    training_data = dataset
    print(f'\nFULL TRAINING MODE: Using all {len(training_data)} rows')

formatted_dataset = training_data.map(
    format_as_chat, remove_columns=training_data.column_names
)

print(f'\nDataset converted to chat format! {len(formatted_dataset)} examples ready.')
print('\nExample of formatted text (first row):')
print('=' * 60)
print(formatted_dataset[0]['text'][:800])
print('=' * 60)

In [ ]:
# ==============================================================
# STEP 5: Load Base Model with 4-bit Quantization (QLoRA)
# ==============================================================
# Unsloth loads the model in 4-bit (uses ~5GB instead of 16GB)
# and is 2x faster than standard HuggingFace loading.

import os
# Helps avoid CUDA memory fragmentation on Colab T4
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from unsloth import FastLanguageModel
import torch

BASE_MODEL_NAME = 'meta-llama/Llama-3.1-8B-Instruct'
MAX_SEQ_LENGTH  = 512    # Safe for Colab T4 GPU
LOAD_IN_4BIT    = True   # Enable QLoRA (4-bit quantization)

print(f'Loading base model: {BASE_MODEL_NAME}')
print('This takes 2-4 minutes on first run (downloads ~4.5 GB)...\n')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,          # Auto-detect best dtype
    load_in_4bit   = LOAD_IN_4BIT,
    token          = HF_TOKEN,
)

# Set padding token safely for training/generation
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print('Base model loaded successfully!')
print(f'  Model type:  {type(model).__name__}')
print(f'  Parameters:  ~8 billion')
print(f'  Loaded as:   4-bit quantized (QLoRA-ready)')

if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  GPU Memory:  {used:.1f} GB used / {total:.1f} GB total')

In [ ]:
# ==============================================================
# STEP 5B: Make Training Data Safe for Colab T4 GPU
# ==============================================================
# Previous error reason:
# Some examples became slightly longer than MAX_SEQ_LENGTH = 512 tokens.
# This cell truncates every training example safely before training.

SAFE_MAX_LENGTH = 448  # keep lower than 512 to avoid length mismatch errors

def truncate_text_to_safe_length(example):
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        max_length=SAFE_MAX_LENGTH,
        add_special_tokens=False,
    )
    example["text"] = tokenizer.decode(
        tokenized["input_ids"],
        skip_special_tokens=False,
    )
    return example

print(f"Truncating training examples to max {SAFE_MAX_LENGTH} tokens...")
formatted_dataset = formatted_dataset.map(
    truncate_text_to_safe_length,
    desc="Truncating long examples safely",
)

# Quick safety check on a few examples
sample_check_size = min(20, len(formatted_dataset))
sample_lengths = [
    len(tokenizer(formatted_dataset[i]["text"], add_special_tokens=False)["input_ids"])
    for i in range(sample_check_size)
]

print("Long examples truncated safely.")
print(f"Dataset ready for training: {len(formatted_dataset)} examples")
print(f"Sample max token length after truncation: {max(sample_lengths)} tokens")
print(f"Model max sequence length: {MAX_SEQ_LENGTH} tokens")


In [ ]:
# ==============================================================
# STEP 6: Test BASE MODEL Before Fine-Tuning
# ==============================================================
# We test the model NOW, before any training.
# We save this answer to compare later with the fine-tuned model.

TEST_QUESTION = 'Suggest a 3-day itinerary for Paris for a first-time traveler.'


def generate_answer(mdl, tkn, question, max_new_tokens=400):
    messages = [
        {'role': 'system',  'content': 'You are a helpful AI travel assistant.'},
        {'role': 'user',    'content': question},
    ]
    input_text = tkn.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tkn(input_text, return_tensors='pt').to('cuda')

    with torch.no_grad():
        outputs = mdl.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            temperature    = 0.7,
            do_sample      = True,
            pad_token_id   = tkn.eos_token_id,
        )

    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    return tkn.decode(generated_ids, skip_special_tokens=True).strip()


# Switch to inference mode for faster generation
FastLanguageModel.for_inference(model)

print('Testing BASE MODEL (before fine-tuning)...')
print(f'\nQuestion: {TEST_QUESTION}\n')
print('Generating answer... (may take 30-60 seconds)')

BASE_MODEL_ANSWER = generate_answer(model, tokenizer, TEST_QUESTION)

print('\n' + '=' * 60)
print('BASE MODEL ANSWER (before fine-tuning):')
print('=' * 60)
print(BASE_MODEL_ANSWER)
print('=' * 60)
print('\nBase model answer saved. We will compare this after training.')

In [ ]:
# ==============================================================
# STEP 7: Add LoRA Adapters to the Model
# ==============================================================
# LoRA adds small trainable matrices to specific layers.
# The original model weights stay FROZEN.
# Only the tiny LoRA matrices are trained.
#
# r         = LoRA rank. Higher = more capacity. Start with 16.
# lora_alpha = scaling factor. Usually 2 x rank.
# target_modules = which layers to apply LoRA to.

model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = [
        'q_proj', 'k_proj',
        'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha                   = 32,
    lora_dropout                 = 0.05,
    bias                         = 'none',
    use_gradient_checkpointing   = 'unsloth',
    random_state                 = 42,
    use_rslora                   = False,
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
pct = 100.0 * trainable_params / total_params

print('LoRA adapters added successfully!')
print(f'  Total parameters:     {total_params:,}')
print(f'  Trainable parameters: {trainable_params:,}  ({pct:.2f}%)')
print(f'\nWe only train {pct:.2f}% of the model - that is the power of LoRA!')
print('The other 99%+ stays frozen and preserves general knowledge.')

In [ ]:
# ==============================================================
# STEP 8: Configure and Run Training
# ==============================================================
# Training loss = how wrong the model predictions are.
# Lower loss = model is learning. Watch it decrease as training runs!

import torch
from trl import SFTTrainer
from transformers import TrainingArguments, TrainerCallback


class LossTracker(TrainerCallback):
    def __init__(self):
        self.train_losses = []
        self.train_steps  = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            self.train_losses.append(logs['loss'])
            self.train_steps.append(state.global_step)
            lr = logs.get('learning_rate', 0)
            print(f'  Step {state.global_step:4d} | Loss: {logs["loss"]:.4f} | LR: {lr:.2e}')


loss_tracker = LossTracker()

# Extra safety for training
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

training_args = TrainingArguments(
    output_dir                  = './outputs',
    num_train_epochs            = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    learning_rate               = 2e-4,
    warmup_steps                = 10,
    logging_steps               = 10,
    save_steps                  = 50,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    optim                       = 'adamw_8bit',
    weight_decay                = 0.01,
    lr_scheduler_type           = 'cosine',
    seed                        = 42,
    report_to                   = 'none',
)

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = formatted_dataset,
    dataset_text_field = 'text',
    max_seq_length     = MAX_SEQ_LENGTH,
    args               = training_args,
    callbacks          = [loss_tracker],
)

print('Training Configuration:')
print(f'  Dataset size:     {len(formatted_dataset)} examples')
print(f'  Epochs:           1')
print(f'  Batch size:       1 x 4 gradient accumulation = effective 4')
print(f'  Learning rate:    2e-4')
print(f'  Max seq length:   {MAX_SEQ_LENGTH} tokens')
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print('\nStarting training... Watch the loss decrease!')
print('  (Lower loss = model is learning travel knowledge)')
print('=' * 60)

train_result = trainer.train()

print('=' * 60)
print('\nTraining completed successfully!')
print(f'  Training time: {train_result.metrics["train_runtime"]:.0f} seconds')
print(f'  Final loss:    {train_result.metrics["train_loss"]:.4f}')
print('\nWhat the loss value means:')
print('  Loss > 2.0   = Model is barely learning')
print('  Loss 1.0-2.0 = Model is learning')
print('  Loss 0.5-1.0 = Model is learning well')
print('  Loss < 0.5   = Model has learned the dataset well')

In [ ]:
# ==============================================================
# STEP 9: Plot Training Loss Graph
# ==============================================================
# A good training curve:
#   - Starts high (model does not know travel yet)
#   - Decreases steadily (model is learning)
#   - Levels off (model has learned what it can from the data)

import matplotlib.pyplot as plt
import numpy as np

if loss_tracker.train_steps:
    fig, ax = plt.subplots(figsize=(10, 5))

    ax.plot(
        loss_tracker.train_steps,
        loss_tracker.train_losses,
        color='#2563EB', linewidth=2, marker='o', markersize=4, label='Training Loss'
    )

    if len(loss_tracker.train_losses) > 3:
        window   = max(1, len(loss_tracker.train_losses) // 5)
        smoothed = np.convolve(
            loss_tracker.train_losses, np.ones(window) / window, mode='valid'
        )
        smooth_steps = loss_tracker.train_steps[window - 1:]
        ax.plot(
            smooth_steps[:len(smoothed)], smoothed,
            color='#DC2626', linewidth=2.5, linestyle='--', label='Smoothed Trend'
        )

    ax.set_xlabel('Training Step', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.set_title('Training Loss over Time\n(Loss going DOWN = model is learning!)',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(bottom=0)

    first_loss = loss_tracker.train_losses[0]
    last_loss  = loss_tracker.train_losses[-1]

    ax.annotate(
        f'Start: {first_loss:.3f}',
        xy=(loss_tracker.train_steps[0], first_loss),
        xytext=(10, 10), textcoords='offset points', fontsize=10, color='green'
    )
    ax.annotate(
        f'End: {last_loss:.3f}',
        xy=(loss_tracker.train_steps[-1], last_loss),
        xytext=(-60, 10), textcoords='offset points', fontsize=10, color='red'
    )

    plt.tight_layout()
    plt.savefig('training_loss.png', bbox_inches='tight')
    plt.show()

    improvement = first_loss - last_loss
    print(f'Loss graph saved as training_loss.png')
    print(f'Loss improved by: {improvement:.4f}  ({first_loss:.4f} -> {last_loss:.4f})')
else:
    print('No loss data recorded. Training may not have logged correctly.')

In [ ]:
# ==============================================================
# STEP 10: Save the LoRA Adapter
# ==============================================================
# We save ONLY the LoRA adapter (not the full 8B model).
# The adapter is tiny (10-100 MB) because it only contains
# the small matrices that were trained.
#
# To use later: load base model + this adapter = fine-tuned model

import os

ADAPTER_PATH = './travel-llama-lora'

print(f'Saving LoRA adapter to: {ADAPTER_PATH}')

model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

print('\nLoRA adapter saved successfully!')
print('\nFiles saved:')
if os.path.exists(ADAPTER_PATH):
    for f in sorted(os.listdir(ADAPTER_PATH)):
        fpath = os.path.join(ADAPTER_PATH, f)
        size  = os.path.getsize(fpath) / 1e6
        print(f'  {f:<40}  {size:.2f} MB')

print('\nTo use this adapter in your backend:')
print('  from peft import PeftModel')
print('  model = PeftModel.from_pretrained(base_model, "./travel-llama-lora")')

In [ ]:
# ==============================================================
# STEP 11: Test Fine-Tuned Model After Training
# ==============================================================
# Test the model with the same question as before.
# We save this answer for the side-by-side comparison.

FastLanguageModel.for_inference(model)

print('Testing FINE-TUNED MODEL (after training)...')
print(f'\nQuestion: {TEST_QUESTION}\n')
print('Generating answer... (may take 30-60 seconds)')

FINETUNED_MODEL_ANSWER = generate_answer(model, tokenizer, TEST_QUESTION)

print('\n' + '=' * 60)
print('FINE-TUNED MODEL ANSWER (after training):')
print('=' * 60)
print(FINETUNED_MODEL_ANSWER)
print('=' * 60)
print('\nFine-tuned model answer saved. Running comparison next...')

In [ ]:
# ==============================================================
# STEP 12: COMPARISON - Base Model vs Fine-Tuned Model
# ==============================================================

separator = '=' * 70

print('\n' + '#' * 70)
print('#  COMPARISON: BASE MODEL vs FINE-TUNED MODEL')
print('#' * 70)
print(f'\nTest Question: {TEST_QUESTION}')

print('\n' + separator)
print('BASE MODEL ANSWER (before fine-tuning):')
print(separator)
print(BASE_MODEL_ANSWER)

print('\n' + separator)
print('FINE-TUNED MODEL ANSWER (after fine-tuning on travel data):')
print(separator)
print(FINETUNED_MODEL_ANSWER)

print('\n' + separator)
print('ANALYSIS - What to look for:')
print(separator)
print('''
Fine-tuned model should be BETTER because:

  MORE STRUCTURED:
    The fine-tuned model has learned to organize travel answers
    with Day 1 / Day 2 / Day 3 format, bullet points, etc.

  MORE SPECIFIC:
    It recommends actual landmarks (Eiffel Tower, Louvre, Montmartre)
    instead of generic advice.

  TRAVEL TONE:
    The fine-tuned model speaks like a travel assistant,
    not a generic chatbot.

  PRACTICAL:
    Includes tips like opening hours, best times to visit,
    and transportation advice.

Note: With only 1 epoch on 1000 rows, improvement may be subtle.
      Full dataset training for 2-3 epochs shows much bigger gains!
''')

print(separator)
print('Fine-tuning complete! Your travel AI chatbot adapter is ready.')
print(separator)

In [ ]:
# ==============================================================
# STEP 13: Interactive Test - Ask Your Own Travel Questions!
# ==============================================================
# Edit MY_QUESTION below and run this cell to ask anything.

MY_QUESTION = 'What are the best street foods to try in Bangkok, Thailand?'

print(f'Your question: {MY_QUESTION}')
print('\nGenerating answer...\n')

my_answer = generate_answer(model, tokenizer, MY_QUESTION)

print('=' * 60)
print('Fine-Tuned Travel Assistant Answer:')
print('=' * 60)
print(my_answer)
print('=' * 60)

In [ ]:
# ==============================================================
# STEP 14 (OPTIONAL): Push LoRA Adapter to Hugging Face Hub
# ==============================================================
# Set PUSH_TO_HUB = True and fill in your HF username to enable.
# This lets you load the adapter from anywhere (your backend server).

PUSH_TO_HUB = False                        # <- Set True to enable
HF_USERNAME = 'your-huggingface-username'  # <- Replace with your username
REPO_NAME   = 'travel-llama-lora'

if PUSH_TO_HUB:
    repo_id = f'{HF_USERNAME}/{REPO_NAME}'
    print(f'Pushing adapter to: https://huggingface.co/{repo_id}')

    model.push_to_hub(repo_id, token=HF_TOKEN, private=True)
    tokenizer.push_to_hub(repo_id, token=HF_TOKEN, private=True)

    print(f'\nAdapter pushed to Hugging Face Hub!')
    print(f'  URL: https://huggingface.co/{repo_id}')
    print(f'\nTo load in your backend:')
    print(f'  model = PeftModel.from_pretrained(base_model, "{repo_id}")')
else:
    print('Skipping Hub push (PUSH_TO_HUB = False)')
    print('  Set PUSH_TO_HUB = True and add your username to enable.')
    print(f'  Adapter is saved locally at: {ADAPTER_PATH}')
    print('  Download it from the Colab file browser (left sidebar -> Files)')

---

# Next Steps - Connect to Your Backend and Next.js

Your fine-tuned LoRA adapter is saved. Here is the roadmap to connect it:

## Backend (Python - FastAPI)
```python
from unsloth import FastLanguageModel
from peft import PeftModel

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name   = 'meta-llama/Llama-3.1-8B-Instruct',
    load_in_4bit = True,
)
model = PeftModel.from_pretrained(base_model, './travel-llama-lora')
FastLanguageModel.for_inference(model)

@app.post('/chat')
def chat(question: str):
    answer = generate_answer(model, tokenizer, question)
    return {'answer': answer}
```

## Frontend (Your Next.js App)
```typescript
const response = await fetch('http://your-backend/chat', {
  method: 'POST',
  body: JSON.stringify({ question: userMessage }),
});
const { answer } = await response.json();
```

## Full Flow
```
User types -> Next.js UI -> POST /chat -> FastAPI Backend
                                              |
                                   LLaMA 3.1 + LoRA Adapter
                                              |
User sees answer <- Next.js UI <- JSON response
```

---

# Improve Your Model - Next Training Steps

| What to try | How | Expected result |
|-------------|-----|-----------------|
| More data | Set `USE_SMALL_DATASET = False` | Better accuracy |
| More epochs | Change `num_train_epochs = 3` | More fluent answers |
| Larger LoRA rank | Change `r = 32` or `r = 64` | Richer responses |
| Lower learning rate | Change to `1e-4` | More stable training |

---
*Fine-tuning notebook for AI Travel Assistant Chatbot - LLaMA 3.1-8B-Instruct + QLoRA*